# 02 Preprocessing

In [1]:
# Imports and setup

from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COL = "DEATH_EVENT"

# This lets the notebook work whether I run it from the notebooks folder or the main project folder.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"

# These folders are needed later when I save tables and results.
for folder in [RESULTS_DIR, TABLES_DIR, METRICS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)


Project root: c:\Users\salem\Desktop\CS439\CS439-Project
Data directory: c:\Users\salem\Desktop\CS439\CS439-Project\data
Results directory: c:\Users\salem\Desktop\CS439\CS439-Project\results


In [2]:
# Load the dataset

# I added a few possible paths so the notebook still works if the CSV is in a slightly different place.
possible_data_paths = [
    DATA_DIR / "heart_failure_clinical_records_dataset.csv",
    PROJECT_ROOT / "heart_failure_clinical_records_dataset.csv",
    CURRENT_DIR / "heart_failure_clinical_records_dataset.csv",
    Path("/mnt/data/heart_failure_clinical_records_dataset.csv"),
]

DATA_PATH = None
for path in possible_data_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "I could not find the heart failure dataset. "
        "Put heart_failure_clinical_records_dataset.csv inside the data/ folder."
    )

df = pd.read_csv(DATA_PATH)

print("Loaded file:", DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


Loaded file: c:\Users\salem\Desktop\CS439\CS439-Project\data\heart_failure_clinical_records_dataset.csv
Dataset shape: (299, 13)


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [3]:
# Basic data checks

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget counts:")
print(df[TARGET_COL].value_counts())

print("\nTarget proportions:")
print(df[TARGET_COL].value_counts(normalize=True).round(4))


Columns:
['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT']

Data types:
age                         float64
anaemia                       int64
creatinine_phosphokinase      int64
diabetes                      int64
ejection_fraction             int64
high_blood_pressure           int64
platelets                   float64
serum_creatinine            float64
serum_sodium                  int64
sex                           int64
smoking                       int64
time                          int64
DEATH_EVENT                   int64
dtype: object

Missing values per column:
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex  

In [4]:
# Clean the dataset

# I keep a copy first so the original dataframe is not changed directly.
df_clean = df.copy()

# Remove exact duplicate rows if there are any.
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
removed_rows = initial_rows - len(df_clean)

print("Rows before cleaning:", initial_rows)
print("Rows after cleaning:", len(df_clean))
print("Duplicate rows removed:", removed_rows)

# Make sure the target column is actually in the dataset.
if TARGET_COL not in df_clean.columns:
    raise ValueError(f"Target column {TARGET_COL} was not found in the dataset.")

# I check missing values before modeling.
missing_total = df_clean.isna().sum().sum()
print("Total missing values:", missing_total)

if missing_total > 0:
    print("Missing values were found. I need to add an imputation step before modeling.")
else:
    print("No missing values found.")


Rows before cleaning: 299
Rows after cleaning: 299
Duplicate rows removed: 0
Total missing values: 0
No missing values found.


## Feature Definitions

The dataset has continuous clinical variables and binary variables.

For my **Logistic Regression** model, I scale the continuous variables because Logistic Regression can be affected by feature scale.

For **Random Forest**, **Decision Tree**, and **Gradient Boosting**, scaling is not needed because tree-based models split based on thresholds.


In [5]:
# Define the target and feature columns

binary_cols = [
    "anaemia",
    "diabetes",
    "high_blood_pressure",
    "sex",
    "smoking",
]

continuous_cols_with_time = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium",
    "time",
]

continuous_cols_without_time = [
    col for col in continuous_cols_with_time if col != "time"
]

feature_cols_with_time = continuous_cols_with_time + binary_cols
feature_cols_without_time = continuous_cols_without_time + binary_cols

# Check that the columns I plan to use are actually in the dataset.
expected_cols = feature_cols_with_time + [TARGET_COL]
missing_cols = [col for col in expected_cols if col not in df_clean.columns]

if missing_cols:
    raise ValueError(f"These expected columns are missing from the dataset: {missing_cols}")

X_with_time = df_clean[feature_cols_with_time].copy()
X_without_time = df_clean[feature_cols_without_time].copy()
y = df_clean[TARGET_COL].copy()

print("Features with time:", X_with_time.shape)
print("Features without time:", X_without_time.shape)
print("Target shape:", y.shape)


Features with time: (299, 12)
Features without time: (299, 11)
Target shape: (299,)


In [6]:
# Stratified train/test split

# I split the row indices once and reuse those same rows for both feature versions.
# This keeps the with-time and without-time comparison fair.

train_idx, test_idx = train_test_split(
    df_clean.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# With time
X_train_with_time = X_with_time.loc[train_idx].reset_index(drop=True)
X_test_with_time = X_with_time.loc[test_idx].reset_index(drop=True)

# Without time
X_train_without_time = X_without_time.loc[train_idx].reset_index(drop=True)
X_test_without_time = X_without_time.loc[test_idx].reset_index(drop=True)

# The target split stays the same for both feature versions.
y_train = y.loc[train_idx].reset_index(drop=True)
y_test = y.loc[test_idx].reset_index(drop=True)

print("X_train_with_time:", X_train_with_time.shape)
print("X_test_with_time:", X_test_with_time.shape)
print("X_train_without_time:", X_train_without_time.shape)
print("X_test_without_time:", X_test_without_time.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting target proportions:")
print(y_test.value_counts(normalize=True).round(4))


X_train_with_time: (239, 12)
X_test_with_time: (60, 12)
X_train_without_time: (239, 11)
X_test_without_time: (60, 11)
y_train: (239,)
y_test: (60,)

Training target proportions:
DEATH_EVENT
0    0.6778
1    0.3222
Name: proportion, dtype: float64

Testing target proportions:
DEATH_EVENT
0    0.6833
1    0.3167
Name: proportion, dtype: float64


In [7]:
# Helper function for Logistic Regression scaling

def scale_for_logistic_regression(X_train, X_test, continuous_cols, binary_cols):
    """
    Scale continuous columns for Logistic Regression and keep binary columns unchanged.
    """
    scaler = StandardScaler()

    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    # Fit the scaler on training data only, then apply it to the test data.
    X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
    X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

    # Keep the column order clean and consistent.
    ordered_cols = continuous_cols + binary_cols
    X_train_scaled = X_train_scaled[ordered_cols]
    X_test_scaled = X_test_scaled[ordered_cols]

    return X_train_scaled, X_test_scaled, scaler


In [8]:
# My Logistic Regression data WITH time

X_train_lr_with_time, X_test_lr_with_time, scaler_lr_with_time = scale_for_logistic_regression(
    X_train=X_train_with_time,
    X_test=X_test_with_time,
    continuous_cols=continuous_cols_with_time,
    binary_cols=binary_cols,
)

print("Logistic Regression WITH time")
print("X_train_lr_with_time:", X_train_lr_with_time.shape)
print("X_test_lr_with_time:", X_test_lr_with_time.shape)
X_train_lr_with_time.head()


Logistic Regression WITH time
X_train_lr_with_time: (239, 12)
X_test_lr_with_time: (60, 12)


,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium,time,anaemia,diabetes,high_blood_pressure,sex,smoking
0,-0.269050,-0.200735,0.176528,-1.004722,-0.360437,0.559915,-0.467847,1,0,0,0,0
1,-0.706883,-0.534318,1.847425,1.051685,-0.544467,-0.345802,-1.359167,0,1,0,1,0
2,1.219579,-0.020580,-1.494369,0.013401,0.467698,-1.477949,-1.591685,0,0,1,1,0
3,0.256348,-0.455129,-1.076645,-0.178127,0.927773,-0.345802,1.121028,0,0,0,1,0
4,-1.407414,-0.020580,-1.494369,-1.387778,0.191653,-0.345802,0.681827,0,0,1,1,0


In [9]:
# My Logistic Regression data WITHOUT time

X_train_lr_without_time, X_test_lr_without_time, scaler_lr_without_time = scale_for_logistic_regression(
    X_train=X_train_without_time,
    X_test=X_test_without_time,
    continuous_cols=continuous_cols_without_time,
    binary_cols=binary_cols,
)

print("Logistic Regression WITHOUT time")
print("X_train_lr_without_time:", X_train_lr_without_time.shape)
print("X_test_lr_without_time:", X_test_lr_without_time.shape)
X_train_lr_without_time.head()


Logistic Regression WITHOUT time
X_train_lr_without_time: (239, 11)
X_test_lr_without_time: (60, 11)


,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium,anaemia,diabetes,high_blood_pressure,sex,smoking
0,-0.269050,-0.200735,0.176528,-1.004722,-0.360437,0.559915,1,0,0,0,0
1,-0.706883,-0.534318,1.847425,1.051685,-0.544467,-0.345802,0,1,0,1,0
2,1.219579,-0.020580,-1.494369,0.013401,0.467698,-1.477949,0,0,1,1,0
3,0.256348,-0.455129,-1.076645,-0.178127,0.927773,-0.345802,0,0,0,1,0
4,-1.407414,-0.020580,-1.494369,-1.387778,0.191653,-0.345802,0,0,1,1,0


In [10]:
# My Random Forest data

X_train_rf_with_time = X_train_with_time.copy()
X_test_rf_with_time = X_test_with_time.copy()

X_train_rf_without_time = X_train_without_time.copy()
X_test_rf_without_time = X_test_without_time.copy()

print("Random Forest WITH time:", X_train_rf_with_time.shape, X_test_rf_with_time.shape)
print("Random Forest WITHOUT time:", X_train_rf_without_time.shape, X_test_rf_without_time.shape)
X_train_rf_with_time.head()


Random Forest WITH time: (239, 12) (60, 12)
Random Forest WITHOUT time: (239, 11) (60, 11)


,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium,time,anaemia,diabetes,high_blood_pressure,sex,smoking
0,58.0,400,40,164000.0,1.0,139,91,1,0,0,0,0
1,53.0,63,60,368000.0,0.8,135,22,0,1,0,1,0
2,75.0,582,20,265000.0,1.9,130,4,0,0,1,1,0
3,64.0,143,25,246000.0,2.4,135,214,0,0,0,1,0
4,45.0,582,20,126000.0,1.6,135,180,0,0,1,1,0


In [11]:
# Tala's Decision Tree and Gradient Boosting data

# Decision Tree data
X_train_tree_with_time = X_train_with_time.copy()
X_test_tree_with_time = X_test_with_time.copy()

X_train_tree_without_time = X_train_without_time.copy()
X_test_tree_without_time = X_test_without_time.copy()

# Gradient Boosting data
X_train_gb_with_time = X_train_with_time.copy()
X_test_gb_with_time = X_test_with_time.copy()

X_train_gb_without_time = X_train_without_time.copy()
X_test_gb_without_time = X_test_without_time.copy()

print("Decision Tree WITH time:", X_train_tree_with_time.shape, X_test_tree_with_time.shape)
print("Decision Tree WITHOUT time:", X_train_tree_without_time.shape, X_test_tree_without_time.shape)
print("Gradient Boosting WITH time:", X_train_gb_with_time.shape, X_test_gb_with_time.shape)
print("Gradient Boosting WITHOUT time:", X_train_gb_without_time.shape, X_test_gb_without_time.shape)


Decision Tree WITH time: (239, 12) (60, 12)
Decision Tree WITHOUT time: (239, 11) (60, 11)
Gradient Boosting WITH time: (239, 12) (60, 12)
Gradient Boosting WITHOUT time: (239, 11) (60, 11)


In [12]:
# Save split indices and preprocessing summary

split_info = pd.DataFrame({
    "original_index": list(train_idx) + list(test_idx),
    "split": ["train"] * len(train_idx) + ["test"] * len(test_idx),
})

split_info_path = TABLES_DIR / "train_test_split_indices.csv"
split_info.to_csv(split_info_path, index=False)

preprocessing_summary = pd.DataFrame({
    "item": [
        "random_state",
        "test_size",
        "rows_after_cleaning",
        "features_with_time",
        "features_without_time",
        "train_rows",
        "test_rows",
        "target_column",
    ],
    "value": [
        RANDOM_STATE,
        TEST_SIZE,
        len(df_clean),
        X_with_time.shape[1],
        X_without_time.shape[1],
        len(y_train),
        len(y_test),
        TARGET_COL,
    ],
})

summary_path = TABLES_DIR / "preprocessing_summary.csv"
preprocessing_summary.to_csv(summary_path, index=False)

print("Saved split information to:", split_info_path)
print("Saved preprocessing summary to:", summary_path)
preprocessing_summary


Saved split information to: c:\Users\salem\Desktop\CS439\CS439-Project\results\tables\train_test_split_indices.csv
Saved preprocessing summary to: c:\Users\salem\Desktop\CS439\CS439-Project\results\tables\preprocessing_summary.csv


,item,value
0,random_state,42
1,test_size,0.2
2,rows_after_cleaning,299
3,features_with_time,12
4,features_without_time,11
5,train_rows,239
6,test_rows,60
7,target_column,DEATH_EVENT


In [13]:
# Save model-ready datasets

# My Logistic Regression files
X_train_lr_with_time.to_csv(TABLES_DIR / "X_train_lr_with_time.csv", index=False)
X_test_lr_with_time.to_csv(TABLES_DIR / "X_test_lr_with_time.csv", index=False)
X_train_lr_without_time.to_csv(TABLES_DIR / "X_train_lr_without_time.csv", index=False)
X_test_lr_without_time.to_csv(TABLES_DIR / "X_test_lr_without_time.csv", index=False)

# My Random Forest files
X_train_rf_with_time.to_csv(TABLES_DIR / "X_train_rf_with_time.csv", index=False)
X_test_rf_with_time.to_csv(TABLES_DIR / "X_test_rf_with_time.csv", index=False)
X_train_rf_without_time.to_csv(TABLES_DIR / "X_train_rf_without_time.csv", index=False)
X_test_rf_without_time.to_csv(TABLES_DIR / "X_test_rf_without_time.csv", index=False)

# Tala's Decision Tree files
X_train_tree_with_time.to_csv(TABLES_DIR / "X_train_tree_with_time.csv", index=False)
X_test_tree_with_time.to_csv(TABLES_DIR / "X_test_tree_with_time.csv", index=False)
X_train_tree_without_time.to_csv(TABLES_DIR / "X_train_tree_without_time.csv", index=False)
X_test_tree_without_time.to_csv(TABLES_DIR / "X_test_tree_without_time.csv", index=False)

# Tala's Gradient Boosting files
X_train_gb_with_time.to_csv(TABLES_DIR / "X_train_gb_with_time.csv", index=False)
X_test_gb_with_time.to_csv(TABLES_DIR / "X_test_gb_with_time.csv", index=False)
X_train_gb_without_time.to_csv(TABLES_DIR / "X_train_gb_without_time.csv", index=False)
X_test_gb_without_time.to_csv(TABLES_DIR / "X_test_gb_without_time.csv", index=False)

# Same labels for everyone
y_train.to_csv(TABLES_DIR / "y_train.csv", index=False)
y_test.to_csv(TABLES_DIR / "y_test.csv", index=False)

print("Saved model-ready preprocessing files to:", TABLES_DIR)


Saved model-ready preprocessing files to: c:\Users\salem\Desktop\CS439\CS439-Project\results\tables


In [14]:
# Final sanity checks

checks = {
    "my_lr_rf_train_rows_match": len(X_train_lr_with_time) == len(X_train_rf_with_time) == len(y_train),
    "my_lr_rf_test_rows_match": len(X_test_lr_with_time) == len(X_test_rf_with_time) == len(y_test),
    "without_time_train_rows_match": len(X_train_lr_without_time) == len(X_train_rf_without_time) == len(y_train),
    "without_time_test_rows_match": len(X_test_lr_without_time) == len(X_test_rf_without_time) == len(y_test),
    "tala_train_rows_match": len(X_train_tree_with_time) == len(X_train_gb_with_time) == len(y_train),
    "tala_test_rows_match": len(X_test_tree_with_time) == len(X_test_gb_with_time) == len(y_test),
    "time_removed_from_without_time": "time" not in X_train_lr_without_time.columns,
    "time_present_in_with_time": "time" in X_train_lr_with_time.columns,
}

for check_name, passed in checks.items():
    print(f"{check_name}: {'PASSED' if passed else 'FAILED'}")

if all(checks.values()):
    print("\nAll preprocessing checks passed. Next step: 03_models.ipynb.")
else:
    raise ValueError("One or more preprocessing checks failed. I need to fix this before modeling.")


my_lr_rf_train_rows_match: PASSED
my_lr_rf_test_rows_match: PASSED
without_time_train_rows_match: PASSED
without_time_test_rows_match: PASSED
tala_train_rows_match: PASSED
tala_test_rows_match: PASSED
time_removed_from_without_time: PASSED
time_present_in_with_time: PASSED

All preprocessing checks passed. Next step: 03_models.ipynb.
